# Step 2: Preprocessing
**INT 396 — Unsupervised Learning | Social Media Topic Discovery for Disaster Response**

Light normalization: remove URLs, @mentions, RT markers, exact duplicates. Normalize hashtags (strip # but keep text). Filter CrisisBench to English-only and remove "not_humanitarian" rows.

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = str(PROJECT_ROOT)

# Load raw datasets
humaid_df = pd.read_parquet(f"{PROJECT_ROOT}/data/raw/humaid_raw.parquet")
cb_df = pd.read_parquet(f"{PROJECT_ROOT}/data/raw/crisisbench_raw.parquet")
print(f"HumAID raw: {len(humaid_df):,} rows")
print(f"CrisisBench raw: {len(cb_df):,} rows")

HumAID raw: 76,484 rows
CrisisBench raw: 141,533 rows


## 2.1 Text Cleaning Functions

In [2]:
def clean_tweet(text):
    """Light normalization for disaster tweets."""
    if not isinstance(text, str):
        return ""
    # Remove RT markers
    text = re.sub(r'^RT\s+@\w+:\s*', '', text)
    # Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Normalize hashtags: strip # but keep the text
    text = re.sub(r'#(\w+)', r'\1', text)
    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Test
sample = "RT @FEMA: #Hurricane damage in #Florida https://t.co/abc123 @RedCross please help!"
print(f"Before: {sample}")
print(f"After:  {clean_tweet(sample)}")

Before: RT @FEMA: #Hurricane damage in #Florida https://t.co/abc123 @RedCross please help!
After:  Hurricane damage in Florida please help!


## 2.2 Process HumAID

In [3]:
# Clean text
humaid_df["text_clean"] = humaid_df["tweet_text"].apply(clean_tweet)

# Remove empty texts after cleaning
before = len(humaid_df)
humaid_df = humaid_df[humaid_df["text_clean"].str.len() > 0].copy()
print(f"Removed {before - len(humaid_df)} empty tweets after cleaning")

# Remove exact duplicates on cleaned text
before = len(humaid_df)
humaid_df = humaid_df.drop_duplicates(subset="text_clean", keep="first").copy()
print(f"Removed {before - len(humaid_df):,} exact duplicates")
print(f"HumAID after cleaning: {len(humaid_df):,} rows")

# Check text length distribution
print(f"\nAvg cleaned length: {humaid_df['text_clean'].str.len().mean():.0f} chars")
print(f"Min: {humaid_df['text_clean'].str.len().min()}, Max: {humaid_df['text_clean'].str.len().max()}")

Removed 0 empty tweets after cleaning
Removed 7 exact duplicates
HumAID after cleaning: 76,477 rows

Avg cleaned length: 132 chars
Min: 11, Max: 368


## 2.3 Process CrisisBench

In [4]:
# Filter to English only
before = len(cb_df)
cb_df = cb_df[cb_df["lang"] == "en"].copy()
print(f"Filtered to English: {before:,} -> {len(cb_df):,} rows (removed {before - len(cb_df):,})")

# Remove "not_humanitarian" rows
before = len(cb_df)
cb_df = cb_df[cb_df["class_label"] != "not_humanitarian"].copy()
print(f"Removed not_humanitarian: {before:,} -> {len(cb_df):,} rows (removed {before - len(cb_df):,})")

# Clean text
cb_df["text_clean"] = cb_df["text"].apply(clean_tweet)

# Remove empty texts
before = len(cb_df)
cb_df = cb_df[cb_df["text_clean"].str.len() > 0].copy()
print(f"Removed {before - len(cb_df)} empty tweets after cleaning")

# Remove exact duplicates on cleaned text
before = len(cb_df)
cb_df = cb_df.drop_duplicates(subset="text_clean", keep="first").copy()
print(f"Removed {before - len(cb_df):,} exact duplicates")
print(f"CrisisBench after cleaning: {len(cb_df):,} rows")

Filtered to English: 141,533 -> 132,619 rows (removed 8,914)
Removed not_humanitarian: 132,619 -> 80,984 rows (removed 51,635)
Removed 0 empty tweets after cleaning
Removed 2 exact duplicates
CrisisBench after cleaning: 80,982 rows


In [5]:
# Verify the 3 pre-selected CrisisBench validation events
target_events = ["2015_nepal_earthquake", "hurricane_harvey", "2013_alberta_floods-ontopic"]
print("Pre-selected validation events:")
for event in target_events:
    count = len(cb_df[cb_df["event"] == event])
    print(f"  {event}: {count:,} tweets")

print(f"\nLabel distribution after filtering:")
print(cb_df["class_label"].value_counts())

Pre-selected validation events:
  2015_nepal_earthquake: 5,236 tweets
  hurricane_harvey: 3,022 tweets
  2013_alberta_floods-ontopic: 4,159 tweets

Label distribution after filtering:
class_label
other_relevant_information             42169
donation_and_volunteering               7400
requests_or_needs                       6917
sympathy_and_support                    5108
infrastructure_and_utilities_damage     5056
affected_individual                     3514
caution_and_advice                      2993
injured_or_dead_people                  2777
disease_related                         1475
response_efforts                        1114
personal_update                         1037
missing_and_found_people                 531
displaced_and_evacuations                511
physical_landslide                       364
terrorism_related                         16
Name: count, dtype: int64


## 2.4 Save Processed Datasets

In [6]:
# Save processed datasets
humaid_df.to_parquet(f"{PROJECT_ROOT}/data/processed/humaid_clean.parquet", index=False)
print(f"Saved: {PROJECT_ROOT}/data/processed/humaid_clean.parquet ({len(humaid_df):,} rows)")

cb_df.to_parquet(f"{PROJECT_ROOT}/data/processed/crisisbench_clean.parquet", index=False)
print(f"Saved: {PROJECT_ROOT}/data/processed/crisisbench_clean.parquet ({len(cb_df):,} rows)")

# Also save the 3 validation event subsets separately
for event in target_events:
    event_df = cb_df[cb_df["event"] == event].copy()
    fname = event.replace("-ontopic", "").replace(" ", "_")
    event_df.to_parquet(f"{PROJECT_ROOT}/data/processed/cb_{fname}.parquet", index=False)
    print(f"Saved: {PROJECT_ROOT}/data/processed/cb_{fname}.parquet ({len(event_df):,} rows)")

Saved: /Users/cyril/Desktop/Un-supervised/data/processed/humaid_clean.parquet (76,477 rows)
Saved: /Users/cyril/Desktop/Un-supervised/data/processed/crisisbench_clean.parquet (80,982 rows)
Saved: /Users/cyril/Desktop/Un-supervised/data/processed/cb_2015_nepal_earthquake.parquet (5,236 rows)
Saved: /Users/cyril/Desktop/Un-supervised/data/processed/cb_hurricane_harvey.parquet (3,022 rows)
Saved: /Users/cyril/Desktop/Un-supervised/data/processed/cb_2013_alberta_floods.parquet (4,159 rows)


## 2.5 Summary

Preprocessing complete. Cleaned datasets saved to `data/processed/`.

**Next step:** Run notebook `03_eda.ipynb` for exploratory data analysis — tweet length distributions, top hashtags, label breakdowns, and event volume analysis.